# Generate Pretend Listening Data
endTime, artistName, trackName, msPlayed

In [1]:
%matplotlib inline

In [2]:
import pandas as pd
import numpy as np

# Random data generator
np.random.seed(42)

artists = ['Type O Negative', 'Incubus', 'Nine Inch Nails', 'Pantera']
tracks = {
    'Type O Negative': ['Stay Out of My Dreams', 'These Three Things', 'World Coming Down', 'Blood & Fire'],
    'Incubus': ['Stellar', 'Glass', 'Crowded Elevator', 'Nice To Know You', 'Just A Phase'],
    'Nine Inch Nails': ['Closer', 'In Two', 'Heresy', 'Vessel', 'Reptile'],
    'Pantera': ['Drag the Waters', '5 Minutes Alone', 'Psycho Holiday', 'Becoming', 'Avoid The Light']
}
    

# Pandas dataframe of the dates, using 2024 calendar year
dates = pd.date_range('2024-01-01', '2024-12-31', freq='h')

# Empty list of records to be populated
records = []

# Loop 300 times to create data
for i in range(3000):
    # Select random artist
    artist = np.random.choice(artists)
    records.append({
        'endTime': str(np.random.choice(dates)),
        'artistName': artist,
        'trackName': np.random.choice(tracks[artist]),
        'msPlayed': np.random.randint(30000, 300000)
    })

df = pd.DataFrame(records).sort_values('endTime').reset_index(drop=True)
# Convert the dataframe to a csv file, index=False says do not write the index as a column in the csv
df.to_json('StreamingHistory.json', orient='records', indent=2)
# Displays top 5 results
df.head()

,endTime,artistName,trackName,msPlayed
0,2024-01-01T01:00:00.000000,Incubus,Just A Phase,94324
1,2024-01-01T02:00:00.000000,Nine Inch Nails,Heresy,52010
2,2024-01-01T04:00:00.000000,Pantera,Becoming,227484
3,2024-01-01T07:00:00.000000,Nine Inch Nails,Vessel,124527
4,2024-01-01T09:00:00.000000,Type O Negative,Stay Out of My Dreams,117958


In [3]:
df = pd.read_json("StreamingHistory.json")

#Convert the endtime to a datetime value
df['endTime'] = pd.to_datetime(df['endTime'])

#Extract the month
df['month'] = df['endTime'].dt.to_period('M')

print(df.dtypes)
df.describe()

# Continue to count listens per month


endTime       datetime64[us]
artistName               str
trackName                str
msPlayed               int64
month              period[M]
dtype: object


,endTime,msPlayed
count,3000,3000.000000
mean,2024-07-02 15:02:07.200000,167082.420667
min,2024-01-01 01:00:00,30215.000000
25%,2024-03-28 07:15:00,99042.250000
50%,2024-07-08 01:30:00,167834.500000
75%,2024-10-03 11:00:00,235036.000000
max,2024-12-30 14:00:00,299756.000000
std,NaN,78039.423043


In [4]:
# For visualizations
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets #interactive UI elements (dropdowns, sliders, etc)
from IPython.display import display # for rendering

# Drop down to select a month
artist_dropdown = widgets.Dropdown(
    options = ['All'] + sorted(df['artistName'].unique().tolist()),
    description='Artist:',
    value='All'
)

def update_chart(artist):
    if artist == 'All':
        filtered = df
    else:
        filtered = df[df['artistName'] == artist]

    monthly = filtered.groupby('month')['trackName'].count()

    fig, ax = plt.subplots(figsize=(12,4))
    
    monthly.plot(kind='bar', ax=ax, color='purple', edgecolor='blue')
    ax.set_title(f'Monthly Listens- {artist}')
    ax.set_xlabel('Month')
    ax.set_ylabel('Total Listens')

    plt.tight_layout()
    plt.show()

widgets.interact(update_chart, artist=artist_dropdown)

interactive(children=(Dropdown(description='Artist:', options=('All', 'Incubus', 'Nine Inch Nails', 'Pantera',…

<function __main__.update_chart(artist)>

In [5]:
def show_summary(artist):
    if artist == 'All':
        filtered = df
    else:
        filtered = df[df['artistName'] == artist]

    print(f"Total Listens:\t {len(filtered)}")
    mostActiveMonth = filtered.groupby('month')['trackName'].count().idxmax()
    print(f"Most Active Month: {mostActiveMonth}")
    mostPlayedTrack = filtered.groupby('trackName')['trackName'].count().idxmax()
    print(f"Most Played Track: {mostPlayedTrack}")

widgets.interact(show_summary, artist=artist_dropdown)

interactive(children=(Dropdown(description='Artist:', options=('All', 'Incubus', 'Nine Inch Nails', 'Pantera',…

<function __main__.show_summary(artist)>